# %% [markdown]
# # STEP 0: Overview
# # Description:
# # This script processes ESP-r heat load files ONLY for specific model folders,
# # mirroring the filtering logic used in your simulation script.
# #
# # Key behaviour:
# # 1. Loops through all building folders in baseline_models
# # 2. ONLY selects model folders matching a predefined set (e.g., SemiDetached_2F_fab, Detached_2F_fab)
# # 3. Navigates directly to their cfg directories
# # 4. Processes heat_load_raw.csv → heat_load.csv
# #
# # This avoids recursive searching and gives you explicit control over which
# # model types are processed.

# %%

In [1]:
# %%
# STEP 1: Setup paths and import libraries
# Description: Define base directory and required modules

import os
import pandas as pd
import re

base_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

print("Base path set successfully")
# %%

Base path set successfully


In [2]:
# %%
# STEP 2: Define valid model folders (CONTROL POINT)
# Description: Only these folders will be processed (same idea as your simulation script)

valid_model_names = {
    "SemiDetached_2F_walls",
    "Detached_2F_walls"
}

print("Valid model folders:", valid_model_names)
# %%

Valid model folders: {'SemiDetached_2F_walls', 'Detached_2F_walls'}


In [3]:
# %%
# STEP 3: Discover model paths
# Description: Replicates your simulation script logic to explicitly select model folders

model_paths = []

for building in os.listdir(base_path):
    building_path = os.path.join(base_path, building)

    if not os.path.isdir(building_path):
        continue

    for sub in os.listdir(building_path):
        sub_path = os.path.join(building_path, sub)

        if sub in valid_model_names and os.path.isdir(sub_path):
            model_paths.append(sub_path)

print(f"Found {len(model_paths)} valid model(s)")
# %%

Found 70 valid model(s)


In [4]:
# %%
# STEP 4: Define heat load processing function
# Description: Cleans and converts ESP-r heat load output

def process_heat_load(input_path, output_path):
    clean_lines = []

    with open(input_path, "r") as file:
        for line in file:
            line = line.strip()

            # Skip comments and headers
            if not line or line.startswith("#") or line.startswith("Time"):
                continue

            clean_lines.append(line)

    data = []

    for line in clean_lines:
        parts = re.split(r"\s+", line)

        time_raw = parts[0]
        flr0_kw = float(parts[1])
        flr1_kw = float(parts[2])
        attic_kw = float(parts[3])

        data.append([time_raw, flr0_kw, flr1_kw, attic_kw])

    df = pd.DataFrame(data, columns=["Time", "flr0_kw", "flr1_kw", "attic_kw"])

    # Convert time format
    def convert_time_format(t):
        match = re.match(r"(\d{2})h(\d{2})", t)
        if match:
            hours, minutes = match.groups()
            return f"{hours}:{minutes}:00"
        return t

    df["Time"] = df["Time"].apply(convert_time_format)

    # Convert kW to Watts
    df["flr0_heat_load"] = (df["flr0_kw"] * 1000).astype(int)
    df["flr1_heat_load"] = (df["flr1_kw"] * 1000).astype(int)
    df["attic_heat_load"] = (df["attic_kw"] * 1000).astype(int)

    df_final = df[["Time", "flr0_heat_load", "flr1_heat_load", "attic_heat_load"]]

    df_final.to_csv(output_path, index=False)
# %%

In [5]:
# %%
# STEP 5: Process each selected model
# Description: Navigate ONLY through controlled model paths and process files

processed = 0
skipped = 0
errors = []

for model_root in model_paths:

    print("\n" + "="*60)
    print(f"Processing model: {model_root}")

    cfg_dir = os.path.join(model_root, "cfg")

    if not os.path.exists(cfg_dir):
        print("WARNING: cfg directory not found, skipping")
        skipped += 1
        continue

    input_file = os.path.join(cfg_dir, "heat_load_raw.csv")
    output_file = os.path.join(cfg_dir, "heat_load.csv")

    if not os.path.exists(input_file):
        print("WARNING: heat_load_raw.csv not found, skipping")
        skipped += 1
        continue

    try:
        process_heat_load(input_file, output_file)
        print(f"[OK] Processed -> {input_file}")
        processed += 1

    except Exception as e:
        print(f"[ERROR] {model_root}: {e}")
        errors.append((model_root, str(e)))

print("\nProcessing complete")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")
print(f"Errors: {len(errors)}")
# %%


Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063637/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063637/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664941/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664941/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1235057414/Detached_2F_walls


[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829079/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664925/Detached_2F_walls


[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664925/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238907/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238907/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664940/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664940/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234982990/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1234982990/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234806523/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1234806523/Detached_2F_walls/c

[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951902/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234621926/SemiDetached_2F_walls


[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1234621926/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627539/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627539/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627541/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627541/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1236594950/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1236594950/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1002031280/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1002031280/Detached_2F_walls/cfg/heat_load

...

[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1002143349/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627544/Detached_2F_walls


[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627544/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664932/Detached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664932/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829058/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829058/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664935/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664935/SemiDetached_2F

[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870859/Detached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829059/SemiDetached_2F_walls


[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829059/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664934/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664934/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829061/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829061/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238911/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238911/SemiDetached_2F_walls/cfg/heat_load_raw.csv

Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829068/SemiDetached_2F_walls
[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829068/Sem

[OK] Processed -> /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063642/Detached_2F_walls/cfg/heat_load_raw.csv

Processing complete
Processed: 70
Skipped: 0
Errors: 0


In [6]:
# %%
# STEP 6: Review errors (if any)
# Description: Output list of failed models for debugging

if errors:
    print("\nError details:")
    for model, err in errors:
        print(f"{model}: {err}")
else:
    print("No errors encountered")
# %%

No errors encountered
